# Imports

In [ ]:
import json
import os
import platform
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import onnxruntime as ort
import pandas as pd
import torch
from torchvision.models.segmentation import lraspp_mobilenet_v3_large

# Params

In [ ]:
MODEL_DIR = "models"
LOG_ROOT = "logs"
RUN_ID = ""
MODELS = ["drone", "satellite"]
RUNTIMES = ["onnxruntime", "torch"]
THREADS = [1, 2, 3, 4]
N_WARMUP = 5
N_ITERS = 30
COOLDOWN_S = 10
N_CLASSES = 6
TEMP_PATH = "/sys/class/thermal/thermal_zone0/temp"
FREQ_PATH = "/sys/devices/system/cpu/cpu0/cpufreq/scaling_cur_freq"
GOVERNOR_PATH = "/sys/devices/system/cpu/cpu0/cpufreq/scaling_governor"

# Data

In [ ]:
references = {name: dict(np.load(Path(MODEL_DIR) / f"reference-{name}.npz")) for name in MODELS}
{name: ref["x"].shape for name, ref in references.items()}

# Model

In [ ]:
def sensors() -> dict:
    throttled = subprocess.run(["vcgencmd", "get_throttled"], capture_output=True, text=True, check=True).stdout
    return {"temp_c": int(Path(TEMP_PATH).read_text()) / 1000,
            "cpu_freq_mhz": int(Path(FREQ_PATH).read_text()) / 1000,
            "throttled": throttled.strip().split("=")[-1]}

In [ ]:
def system_info() -> dict:
    cpuinfo = Path("/proc/cpuinfo").read_text().splitlines()
    board = next(line.split(":", 1)[1].strip() for line in cpuinfo if line.startswith("Model"))
    return {"board": board, "platform": platform.platform(), "python": platform.python_version(),
            "torch": torch.__version__, "onnxruntime": ort.__version__, "numpy": np.__version__,
            "cpu_count": os.cpu_count(), "governor": Path(GOVERNOR_PATH).read_text().strip(),
            "time_utc": datetime.now(timezone.utc).isoformat(), **sensors()}

In [ ]:
def onnx_runner(name: str, threads: int):
    options = ort.SessionOptions()
    options.intra_op_num_threads = threads
    options.inter_op_num_threads = 1
    session = ort.InferenceSession(str(Path(MODEL_DIR) / f"lraspp-{name}.onnx"), options,
                                   providers=["CPUExecutionProvider"])
    return lambda x: session.run(None, {"image": x})[0]

In [ ]:
def torch_model(name: str) -> torch.nn.Module:
    net = lraspp_mobilenet_v3_large(weights=None, weights_backbone=None, num_classes=N_CLASSES)
    state = torch.load(Path(MODEL_DIR) / f"lraspp-{name}.pt", map_location="cpu")
    net.load_state_dict({key.removeprefix("net."): value for key, value in state.items()})
    return net.eval()

In [ ]:
def torch_runner(name: str, threads: int):
    torch.set_num_threads(threads)
    net = torch_model(name)

    @torch.no_grad()
    def run(x: np.ndarray) -> np.ndarray:
        return net(torch.from_numpy(x))["out"].numpy()
    return run

In [ ]:
RUNNER_FACTORIES = {"onnxruntime": onnx_runner, "torch": torch_runner}

In [ ]:
def time_config(runner, x: np.ndarray, config: dict, log_path: Path) -> list[dict]:
    for _ in range(N_WARMUP):
        runner(x)
    rows = []
    with log_path.open("a") as log:
        for i in range(N_ITERS):
            start = time.perf_counter()
            runner(x)
            latency_ms = (time.perf_counter() - start) * 1000
            rows.append({**config, "iter": i, "latency_ms": latency_ms,
                         "time_utc": datetime.now(timezone.utc).isoformat(), **sensors()})
            log.write(json.dumps(rows[-1]) + "\n")
            log.flush()
    return rows

In [ ]:
def bench(runtime: str, name: str, threads: int, log_path: Path) -> dict:
    runner = RUNNER_FACTORIES[runtime](name, threads)
    reference = references[name]
    config = {"runtime": runtime, "model": name, "threads": threads, "input_shape": list(reference["x"].shape)}
    time_config(runner, reference["x"], config, log_path)
    return {**config, "max_abs_diff": float(np.abs(runner(reference["x"]) - reference["logits"]).max())}

# Run

In [ ]:
log_dir = Path(LOG_ROOT) / (RUN_ID or datetime.now().strftime("%Y%m%d-%H%M%S"))
log_dir.mkdir(parents=True)
latency_log = log_dir / "latency.jsonl"
log_dir

In [ ]:
system_start = system_info()
system_start

In [ ]:
parities = []
for runtime in RUNTIMES:
    for name in MODELS:
        for threads in THREADS:
            parities.append(bench(runtime, name, threads, latency_log))
            print(parities[-1])
            time.sleep(COOLDOWN_S)

In [ ]:
params = {"N_WARMUP": N_WARMUP, "N_ITERS": N_ITERS, "COOLDOWN_S": COOLDOWN_S, "THREADS": THREADS}
(log_dir / "system.json").write_text(json.dumps(
    {"start": system_start, "end": system_info(), "params": params}, indent=2))
(log_dir / "parity.json").write_text(json.dumps(parities, indent=2))

# Summary

In [ ]:
latency = pd.read_json(latency_log, lines=True)
summary = latency.groupby(["runtime", "model", "threads"]).agg(
    median_ms=("latency_ms", "median"), p90_ms=("latency_ms", lambda s: s.quantile(0.9)),
    max_temp_c=("temp_c", "max"), min_freq_mhz=("cpu_freq_mhz", "min"))
summary.to_csv(log_dir / "summary.csv")
summary